In [1]:
# Week 9 Day 1 Exercises XP

# 👩‍🏫 👩🏿‍🏫 What You’ll learn

#     Explore how JavaScript interacts with HTML elements by modifying a simple web page.
#     Familiarize yourself with JavaScript variables and primitive data types.
#     Understand the difference between a JavaScript-driven page and a static HTML page.
#     Scrape movie data from a dynamically loaded page using Selenium and BeautifulSoup.
#     Scrape and analyze hotel reviews from TripAdvisor.


# 🛠️ What you will create

#     A simple HTML page with a JavaScript script that changes text dynamically when a button is clicked.
#     A script showcasing the use of different JavaScript data types and outputting them to the browser’s console.
#     Two HTML files demonstrating the difference in behavior between a static page and one enhanced with JavaScript.
#     A Python script using Selenium and BeautifulSoup to scrape movie information from a dynamically loaded Rotten Tomatoes page.
#     A tool to scrape and categorize book data based on ratings from Amazon’s Best Sellers page.
#     A scraper analyzing hotel reviews from TripAdvisor, including basic sentiment analysis of the review texts.

In [5]:
# 🌟 Exercise 3 : Scrape Dynamic Content from Rotten Tomatoes
# Task:

#     Use Selenium to navigate to the Rotten Tomatoes Certified Fresh Movies page.
#     Extract the HTML content after it’s fully loaded.
#     Use BeautifulSoup to parse and extract the movie titles, scores, and release dates.

# Instructions

#     Set up Selenium WebDriver and navigate to the Rotten Tomatoes page.
#     Extract the HTML content using driver.page_source.
#     Parse the HTML with BeautifulSoup.
#     Find and extract the desired movie information.
#     Print the extracted data.


# Install libraries (google-colab-selenium handles Chrome + driver automatically)
!pip install -q google-colab-selenium beautifulsoup4



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.3 MB/s eta 0:00:00


In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [7]:
# Set up Selenium — google-colab-selenium handles Chrome, driver, and all options!
driver = gs.Chrome()
print("Selenium ready! Headless Chrome is running.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Selenium ready! Headless Chrome is running.


In [16]:
#     Set up Selenium WebDriver and navigate to the Rotten Tomatoes page.
#     Extract the HTML content using driver.page_source.
#     Parse the HTML with BeautifulSoup.
# Full pipeline: JS page → Selenium → BS4 → DataFrame
url = "https://www.rottentomatoes.com/browse/movies_at_home/critics:certified_fresh"
# 1. driver.get(url)
driver.get(url)

# 2. WebDriverWait for the main movie list items, as the page is dynamic
# A more robust selector for the entire movie item block is often better for waiting.
WebDriverWait(driver, 40).until(
    EC.presence_of_all_elements_located((By.TAG_NAME, 'media-info-tile'))
)

# 3. BeautifulSoup(driver.page_source, ...)
soup = BeautifulSoup(driver.page_source, 'html.parser')

# 4. Find and extract the desired movie information.
movie_data = []
# Find all movie list items
movie_items = soup.find_all('media-info-tile')

for item in movie_items:
    # Extract critics score
    critics_score_elem = item.find(slot='criticsScore')
    critics_score = critics_score_elem.get_text(strip=True) if critics_score_elem else 'N/A'

    # Extract audience score
    audience_score_elem = item.find(slot='audienceScore')
    audience_score = audience_score_elem.get_text(strip=True) if audience_score_elem else 'N/A'

    # Extract title using data-qa attribute
    title_elem = item.find('rt-text', attrs={'data-qa': 'discovery-media-list-item-title'})
    title = title_elem.get_text(strip=True) if title_elem else 'N/A'

    # Extract streaming date using data-qa attribute
    streaming_date_elem = item.find('rt-text', attrs={'data-qa': 'discovery-media-list-item-start-date'})
    streaming_date = streaming_date_elem.get_text(strip=True) if streaming_date_elem else 'N/A'

    movie_data.append({
        'criticsScore': critics_score,
        'audienceScore': audience_score,
        'title': title,
        'streamingDate': streaming_date
    })


In [17]:

#     Print the extracted data.
# 5. pd.DataFrame(...)
df_movies = pd.DataFrame(movie_data)
display(df_movies.head(10))

,criticsScore,audienceScore,title,streamingDate
0,94%,95%,Project Hail Mary,"Streaming May 12, 2026"
1,95%,86%,The Christophers,"Streaming May 12, 2026"
2,72%,54%,Hamlet,"Streaming May 12, 2026"
3,82%,90%,Remarkably Bright Creatures,"Streaming May 8, 2026"
4,93%,87%,Send Help,"Streaming Mar 24, 2026"
5,76%,78%,The Drama,"Streaming May 5, 2026"
6,99%,80%,The Perfect Neighbor,"Streaming Oct 17, 2025"
7,93%,86%,Exit 8,"Streaming May 8, 2026"
8,94%,93%,Hoppers,"Streaming Apr 28, 2026"
9,74%,89%,Ready or Not 2: Here I Come,"Streaming May 5, 2026"


In [29]:
# 🌟 Exercise 4 : Scrape and Categorize News Articles from a JavaScript-Enabled News Site
# Task:

#     Visit this website. https://www.bbc.com/technology
#     Scrape news article titles and their publication dates.
#     Categorize articles based on their publication month.

# Instructions:

#     Use Selenium to navigate to a specific news section on the website.
#     Extract and parse the HTML content that is dynamically loaded via JavaScript.
#     Using BeautifulSoup, extract news article titles and publication dates.

# Full pipeline: JS page → Selenium → BS4 → DataFrame
url = "https://www.bbc.com/technology"
# 1. driver.get(url)
driver.get(url)

## News articles links are located in various ways:

## Publication dates are not listed outright. Some are located in the link. So for the purpose of this activity we will use the More section, which shows a relative time or date of publication.
# Each article element is located within a block of:
# <a href="/news/articles/cx219exww6eo" data-testid="internal-link" class="sc-8a623a54-0 huZCWi">
# Can't use that class name. Each article element is also located within:
# <div data-testid="liverpool-card" data-indexcard="true" class="sc-225578b-0 ezQaGx"><div data-testid="anchor-inner-wrapper">
# So can we use data-testid "liverpool-card" or data-testid "anchor-inner-wrapper"?

# The time element is located in:
# <span data-testid="card-metadata-lastupdated" class="sc-1907e52a-1 bKFIy">6 hrs ago</span>

# The Title element is located in:
# <h2 data-testid="card-headline" class="sc-feaf8701-3 dkPqjG">My postpartum body should not be a talking point on social media</h2>


# 2. WebDriverWait for the main movie list items, as the page is dynamic
# A more robust selector for the entire movie item block is often better for waiting.
WebDriverWait(driver, 40).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, '[data-testid="liverpool-card"]'))
)

# 3. BeautifulSoup(driver.page_source, ...)
soup = BeautifulSoup(driver.page_source, 'html.parser')

# 4. Find and extract the desired movie information.
article_data = []
# Find all movie list items
article_items = soup.find_all('div', attrs={'data-testid': 'liverpool-card'})

for item in article_items:
    # Extract time info
    time_info_elem = item.find('span', attrs={'data-testid': 'card-metadata-lastupdated'})
    time_info = time_info_elem.get_text(strip=True) if time_info_elem else 'N/A'

    # Extract title using data-qa attribute
    title_elem = item.find('h2', attrs={'data-testid': 'card-headline'})
    title = title_elem.get_text(strip=True) if title_elem else 'N/A'


    article_data.append({
        'time_info': time_info,
        'title': title
    })


In [33]:
import pandas as pd
from datetime import datetime, timedelta
import re # Import regex module

# Create a DataFrame from the scraped article data
df_articles = pd.DataFrame(article_data)

# Function to parse relative time strings into datetime objects
def parse_relative_time(time_str):
    time_str = time_str.lower().strip() # Normalize string
    now = datetime.now()

    # Try matching minutes
    match = re.match(r'(\d+)\s+mins? ago', time_str)
    if match:
        num = int(match.group(1))
        return now - timedelta(minutes=num)

    # Try matching hours
    match = re.match(r'(\d+)\s+hrs? ago', time_str)
    if match:
        num = int(match.group(1))
        return now - timedelta(hours=num)

    # Try matching days
    match = re.match(r'(\d+)\s+days? ago', time_str)
    if match:
        num = int(match.group(1))
        return now - timedelta(days=num)

    return None # Return None for unparseable strings

# Apply the function to the 'time_info' column to create a new 'publication_date' column
df_articles['publication_date'] = df_articles['time_info'].apply(parse_relative_time).dt.date

# Display the DataFrame with the new 'publication_date' column
display(df_articles.head())

,time_info,title,publication_date
0,26 mins ago,"Suicide forum fined £950,000 for not blocking ...",2026-05-13
1,1 hr ago,Thousands of Waymos recalled after robotaxi sw...,2026-05-13
2,7 hrs ago,My postpartum body should not be a talking poi...,2026-05-13
3,7 hrs ago,'I only talk to my friends on my phone - I don...,2026-05-13
4,17 hrs ago,Elon Musk said control of OpenAI should go to ...,2026-05-12


In [34]:
#     Categorize articles by their publication month (e.g., ‘January’, ‘February’, etc.).
#     Print the categorized lists of articles.

import pandas as pd # Ensure pandas is imported for pd.notnull

df_articles['publication_month'] = df_articles['publication_date'].apply(lambda x: x.strftime('%B') if pd.notnull(x) else 'Unknown Month')

categorized_articles = {}
for month, group in df_articles.groupby('publication_month'):
    categorized_articles[month] = group[['title', 'publication_date']].to_dict('records')

# Print the categorized lists of articles
for month, articles in categorized_articles.items():
    print(f"\n--- {month} ---")
    for article in articles:
        # Ensure 'publication_date' is not NaT before formatting
        formatted_date = article['publication_date'].strftime('%Y-%m-%d') if pd.notnull(article['publication_date']) else 'Date not available'
        print(f"  Title: {article['title']}")
        print(f"  Published: {formatted_date}")


--- May ---
  Title: Suicide forum fined £950,000 for not blocking UK users
  Published: 2026-05-13
  Title: Thousands of Waymos recalled after robotaxi swept into a creek
  Published: 2026-05-13
  Title: My postpartum body should not be a talking point on social media
  Published: 2026-05-13
  Title: 'I only talk to my friends on my phone - I don't meet them'
  Published: 2026-05-13
  Title: Elon Musk said control of OpenAI should go to his children, Sam Altman tells jury
  Published: 2026-05-12
  Title: EU needs to delay social media access for children - von der Leyen
  Published: 2026-05-12
  Title: Texas accuses Netflix of spying on users, including children
  Published: 2026-05-12
  Title: 'Why I'm celebrating unsung hero teachers on TikTok'
  Published: 2026-05-12
  Title: Renewable energy hub planned for Scottish coal museum
  Published: 2026-05-12
  Title: Scrap yard calls for battery removal when recycling
  Published: 2026-05-12
  Title: Dua Lipa sues Samsung for $15m over 

In [24]:
# 🌟 Exercise 5 : Scrape and Analyze Weather Data from a JavaScript-Enabled Weather Website
# Task:

#     Visit this website. 'https://www.accuweather.com/en/us/los-angeles-ca/90012/weather-forecast/348108'
#     Scrape weather forecast data including temperature, condition, and humidity.
#     Analyze the data to find the average temperature and most common weather condition.

# Instructions:

#     Use Selenium to navigate to the weather forecast page of a specific city.
#     Extract and parse the HTML content, focusing on dynamically loaded weather data.
#     Using BeautifulSoup, extract relevant weather information like temperature, condition (sunny, cloudy, etc.), and humidity.

# Using accuweather.com failed, presumably because of anti-scraping protections on the site. Using an alternative site instead, with weather for New Delhi.

import time
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
from bs4 import BeautifulSoup
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import pandas as pd
import re

# New URL for worldweather.wmo.int
new_weather_url = 'https://worldweather.wmo.int/en/city.html?cityId=224'

# Assuming 'driver' is already initialized and active from previous exercises
# If not, ensure gs.Chrome() is called before this cell.

try:
    print(f"Navigating to: {new_weather_url}")
    driver.get(new_weather_url)
    print(f"Current URL: {driver.current_url}")

    # Wait for a prominent element that indicates the page has loaded, e.g., the city name or a forecast item container
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CLASS_NAME, 'city_forecast_day_object'))
    )
    print("Page loaded successfully and forecast elements are present.")

    # Get the page source after dynamic content has loaded
    soup_wmo = BeautifulSoup(driver.page_source, 'html.parser')

except (TimeoutException, WebDriverException) as e:
    print(f"Failed to load or scrape page {new_weather_url}: {e}")
    soup_wmo = BeautifulSoup("", 'html.parser') # Initialize empty soup if error

forecast_data_wmo = []

if soup_wmo and soup_wmo.body: # Ensure soup is not empty
    # Find all daily forecast items using the class provided by the user
    forecast_items = soup_wmo.find_all('div', class_='city_forecast_day_object')

    if not forecast_items:
        print("No forecast items found with the specified selector. Please check the HTML structure.")
    else:
        for item in forecast_items:
            # Extract Date
            date_elem = item.find('div', class_='city_fc_date')
            date_text = date_elem.get_text(strip=True).split('(')[0].strip() if date_elem else 'N/A'

            # Extract Min Temperature
            min_temp_elem = item.find('span', class_='min_temp_box')
            min_temp = min_temp_elem.get_text(strip=True) if min_temp_elem else 'N/A'

            # Extract Max Temperature
            max_temp_elem = item.find('span', class_='max_temp_icon')
            max_temp = max_temp_elem.get_text(strip=True) if max_temp_elem else 'N/A'

            # Extract Condition
            condition_elem = item.find('div', class_='city_fc_desc')
            condition = condition_elem.get_text(strip=True) if condition_elem else 'N/A'

            forecast_data_wmo.append({
                'date': date_text,
                'min_temp': min_temp,
                'max_temp': max_temp,
                'condition': condition
            })

    # Create a DataFrame from the scraped forecast data
    df_forecast_wmo = pd.DataFrame(forecast_data_wmo)
    print('Weather Forecast for New Delhi:')
    display(df_forecast_wmo.head(10))

    # --- Data Analysis ---

    # Function to extract numeric temperature from string (e.g., '27°C' -> 27)
    def extract_numeric_temp(temp_str):
        if isinstance(temp_str, str) and temp_str != 'N/A':
            match = re.search(r'(\d+)', temp_str)
            if match:
                return int(match.group(1))
        return None

    df_forecast_wmo['min_temp_numeric'] = df_forecast_wmo['min_temp'].apply(extract_numeric_temp)
    df_forecast_wmo['max_temp_numeric'] = df_forecast_wmo['max_temp'].apply(extract_numeric_temp)

    # Calculate average min and max temperatures
    avg_min_temp = df_forecast_wmo['min_temp_numeric'].dropna().mean()
    avg_max_temp = df_forecast_wmo['max_temp_numeric'].dropna().mean()

    print(f"\nAverage Minimum Temperature: {avg_min_temp:.2f}°C")
    print(f"Average Maximum Temperature: {avg_max_temp:.2f}°C")

    # Identify the most common weather condition
    if not df_forecast_wmo['condition'].empty:
        most_common_condition = df_forecast_wmo['condition'].mode()[0]
        print(f"Most Common Weather Condition: {most_common_condition}")
    else:
        print("No weather conditions found to determine the most common.")

else:
    print("BeautifulSoup object is empty or invalid. Skipping data extraction and analysis.")

Navigating to: https://worldweather.wmo.int/en/city.html?cityId=224
Current URL: https://worldweather.wmo.int/en/city.html?cityId=224
Page loaded successfully and forecast elements are present.
Weather Forecast for New Delhi:


,date,min_temp,max_temp,condition
0,14 May,27°C,39°C,Partly Cloudy
1,15 May,27°C,40°C,Partly Cloudy
2,16 May,27°C,40°C,Clear
3,17 May,28°C,41°C,Clear
4,18 May,28°C,41°C,Clear
5,19 May,28°C,41°C,Clear



Average Minimum Temperature: 27.50°C
Average Maximum Temperature: 40.33°C
Most Common Weather Condition: Clear
